# CBSL Wage and Producer Price Preprocessing

In [ ]:
import re
import pandas as pd
from paths import PREPROCESSED_PATHS, RAW_PATHS, save_dataset


In [ ]:
def sanitize(*parts):
    text = '_'.join(str(p).strip() for p in parts if str(p).strip())
    text = re.sub(r'[^\w]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

def year_columns(header_row):
    years = []
    for col_idx in header_row.index:
        text = str(header_row[col_idx]).strip().replace('(a)', '').strip()
        try:
            year = int(float(text))
        except ValueError:
            continue
        if 2000 <= year <= 2100:
            years.append((col_idx, year))
    return years

def extract_producer_prices(path):
    raw = pd.read_excel(path, header=None)
    records = []
    in_section = False
    for _, row in raw.iterrows():
        label = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ''
        if label == 'Commodity':
            in_section = True
            continue
        if in_section and label.startswith('('):
            break
        if not in_section or not label.lower().startswith('paddy'):
            continue
        for col_idx, year in year_columns(raw.iloc[3]):
            value = pd.to_numeric(row[col_idx], errors='coerce')
            if pd.notna(value):
                records.append({'Year': year, 'Feature': sanitize('CBSL', 'Producer_Price_Paddy_LKR'), 'Value': value})
    return pd.DataFrame(records)

def extract_paddy_wages(path):
    raw = pd.read_excel(path, header=None)
    records = []
    in_paddy = False
    for _, row in raw.iterrows():
        activity = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ''
        labour = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ''
        if activity == 'Paddy':
            in_paddy = True
            continue
        if in_paddy and activity.startswith('('):
            break
        if not in_paddy or labour != 'M' or not activity or activity == 'nan':
            continue
        for col_idx, year in year_columns(raw.iloc[3]):
            value = pd.to_numeric(row[col_idx], errors='coerce')
            if pd.notna(value):
                records.append({'Year': year, 'Feature': sanitize('CBSL', 'Wage', activity), 'Value': value})
    return pd.DataFrame(records)

def extract_monthly_wages(path):
    raw = pd.read_excel(path, header=None)
    month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
    header = raw.iloc[3]
    month_cols = {idx: month_map[str(label).strip()] for idx, label in header.items() if str(label).strip() in month_map}
    records = []
    current_year = current_crop = current_activity = None
    for _, row in raw.iterrows():
        first = str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else ''
        second = str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else ''
        if first.startswith('Year'):
            match = re.search(r'(\d{4})', first)
            current_year = int(match.group(1)) if match else None
            continue
        if first == 'Agriculture':
            continue
        if first and first != 'nan':
            current_crop = first
            current_activity = None
        if second and second != 'nan':
            current_activity = second
        if current_year is None or current_crop != 'Paddy' or not current_activity:
            continue
        for col_idx, month in month_cols.items():
            value = pd.to_numeric(row[col_idx], errors='coerce')
            if pd.notna(value):
                records.append({'Year': current_year, 'Month': month, 'Feature': sanitize('CBSL', 'MonthlyWage', current_crop, current_activity), 'Value': value})
    monthly = pd.DataFrame(records)
    if monthly.empty:
        return monthly
    monthly['YearMonth'] = pd.to_datetime(monthly['Year'].astype(str) + '-' + monthly['Month'].astype(str) + '-01')
    return monthly.sort_values('YearMonth').groupby(['Year', 'Feature'], as_index=False)['Value'].last()

producer = extract_producer_prices(RAW_PATHS['cbsl_producer_prices'])
wages = extract_paddy_wages(RAW_PATHS['cbsl_daily_wages'])
monthly = extract_monthly_wages(RAW_PATHS['cbsl_monthly_wages'])
combined = pd.concat([producer, wages, monthly], ignore_index=True)
annual = combined.pivot_table(index='Year', columns='Feature', values='Value', aggfunc='mean')
save_dataset(annual, PREPROCESSED_PATHS['cbsl_annual'])
print('CBSL annual shape:', annual.shape)
